# Adversarial Debate — LangChain

**Pattern:** Proposer → Critic → Judge

```
proposer_chain ──FOR──▶ critic_chain ──AGAINST──▶ judge_chain ──▶ Verdict
```

Three LCEL chains with opposing personas. The judge receives both sides and scores them.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.3)
parser = StrOutputParser()
print("✓ Ready")

In [ ]:
proposer_chain = (ChatPromptTemplate.from_messages([
    ("system","Make the STRONGEST case FOR the claim. 3 arguments, 150 words."),
    ("human","Claim: {claim}\n\nArgue FOR:")]) | llm | parser)

critic_chain = (ChatPromptTemplate.from_messages([
    ("system","Argue AGAINST the claim. Find weaknesses. 150 words."),
    ("human","Claim: {claim}\n\nFOR argument:\n{proposal}\n\nArgue AGAINST:")]) | llm | parser)

judge_chain = (ChatPromptTemplate.from_messages([
    ("system","Impartial judge. Score each side 1-10, name stronger side, give conclusion."),
    ("human","Claim: {claim}\n\nFOR:\n{proposal}\n\nAGAINST:\n{critique}\n\nVerdict:")]) | llm | parser)

print("3 debate chains ready")

In [ ]:
claim = "Tokyo is the best travel destination for a one-week trip"

print("[Proposer] arguing FOR...")
proposal = proposer_chain.invoke({"claim": claim})
print(proposal)

print("\n[Critic] arguing AGAINST...")
critique = critic_chain.invoke({"claim": claim, "proposal": proposal})
print(critique)

print("\n[Judge] deliberating...")
verdict = judge_chain.invoke({"claim": claim, "proposal": proposal, "critique": critique})
print(verdict)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Different `temperature=0.3` | Debate benefits from some creativity, not just facts |
| Critic receives the proposal | Critic argues against SPECIFIC points, not generic counter |
| Judge receives both | Verdict is balanced — can't be biased toward one side |

**Why adversarial beats single-LLM:** A single LLM asked to 'critique Tokyo' will be polite. An agent whose sole role is to argue AGAINST will find sharper weaknesses.

### Next: [LangGraph Debate →](../LangGraph/debate.ipynb)